# UNVERIFIED / INCOMPLETE NOTEBOOK

**未検証・途中までの参考Notebookです。** 全セルの実行成功は確認していません。

**Stopping reason:** Reproduction fully validated on fresh CPU Colab runtime (45/45 Spearman rho match published Table S2, max |drho|=5.6e-16; 20/45 significant; headline shape-size independence reproduced; 11/11 cells, 0 errors). Only GitHub publication failed: all pushes to phytometrics/phenopaper-colab-notebooks rejected with remote 'Internal Server Error' (publisher attempts x3, direct push of prepared commit, and a no-op temp-ref probe; auth OK, GitHub status 'Git Operations operational'). Notebook and evidence preserved for a retry when GitHub accepts pushes.

Attempt status: failed. No manual login validation was performed. This draft may require your own authorized access, fail partway, or contain incomplete code. This attempt does not establish that the paper cannot be reproduced.

Paper: Differences in apple fruit shape are independent of fruit size

[PhenoPaper record](https://phenopaper.smartbreed-plant-phenotyping-platform.com/papers/p-76b3fc60078b3d0914bdc7784044dc1e)


# Apple fruit shape vs fruit size — correlation reproduction

**Paper:** *Differences in apple fruit shape are independent of fruit size* — bioRxiv 2025.01.01.631017v1 (posted 2025-01-02), DOI: [10.1101/2025.01.01.631017](https://doi.org/10.1101/2025.01.01.631017); journal version DOI 10.1002/ppj2.70032.
**Author code:** [kylidevi/apple_shape](https://github.com/kylidevi/apple_shape) @ commit `5bf0df08c993d1c59cdcdd52435b5185ecada11a` (MIT-attributed README; analysis in Quarto/R/Python).
**Author data:** [Dryad doi:10.5061/dryad.xgxd254tg](https://doi.org/10.5061/dryad.xgxd254tg) (CC0-1.0) and the bioRxiv Online Supporting Information tables.

**What this notebook does (scope):** it reproduces the paper's central statistical result — the Bonferroni-corrected Spearman correlation matrix among 10 fruit traits (Table S2; the basis of Figures 2/4 and the headline claim that shape PC1 tracks aspect ratio, rho ≈ −0.964, but is *not* significantly correlated with fruit size) — starting from the authors' published per-image results table (`TableS1.csv`, 5724 images × 35 columns, containing their computed linear measurements, PCA scores and REML-adjusted values). It also renders a Figure-3-style PC1×PC3 morphospace colored by aspect ratio.

**What is NOT reproduced here:** the image pipeline (MATLAB colour correction, manual ImageJ top/bottom landmarking, OpenCV contouring to 99 pseudo-landmarks), the generalized Procrustes analysis / PCA refit from binary images, eigen-apples via inverse PCA, and the `lme4` REML orchard-position adjustment. The binary-image archive (`all_binaries.zip`, Dryad) is not retrievable through authorized anonymous automated routes (API download now requires a Dryad bearer token; the UI route serves an anti-bot challenge), so this notebook uses the authors' **published downstream outputs** instead of regenerating them. The REML-adjusted trait columns are taken as published.

**Execution status:** executed end-to-end in Colab (CPU runtime); outputs shown are from the recorded validation run.

**AI-assisted port notice / AI支援の翻訳に関する注意:** The correlation analysis below is an **AI-assisted R→Python translation** of the authors' `TableS2.R` (commit `5bf0df0`): `cor.test(method="spearman")` → `scipy.stats.spearmanr`, `p.adjust(method="bonferroni")` → explicit Bonferroni step, `distinct(apple_id, .keep_all=TRUE)` → `drop_duplicates`. The authors did not provide this Colab implementation. The executed example and listed checks were validated, but untested behavior is not guaranteed.
The morphospace rendering reuses the authors' own Python parameters from `apple_morphometric.qmd` (same axes, colormap and normalization) but omits the inverse-PCA eigen-apples.

**日本語の要約:** このノートブックは、リンゴ果実の形状（PC1）が果実サイズと無相関であるという論文の主要結果を、著者公開の結果表 `TableS1.csv`（5724画像×35列、REML調整済み形質値を含む）から再計算します。著者の R スクリプト `TableS2.R` を Python（SciPy）に翻訳し、45ペアの Spearman 順相関と Bonferroni 補正を適用して、bioRxiv 補足 Table S2（media-4.csv）と照合します。画像処理・GPA・PCA・逆PCAのeigen-apple、REML再推定は範囲外です（Dryad の画像アーカイブは自動ダウンロードがブロックされるため）。

## Runtime selection / ランタイム選択

**Use the CPU runtime** (Runtime → Change runtime type → CPU). This notebook is pure `pandas`/`numpy`/`scipy`/`matplotlib` table analysis; there is no neural network, no CUDA usage, and the whole run takes well under a minute of compute plus a ~2.7 MB download. A GPU adds nothing.

**Run all:** Runtime → Run all. Expected total time ≈ 1 minute (dominated by downloads); downloads ≈ 3 MB total.
**実行方法:** 「ランタイム → すべてのセルを実行」。CPU で約1分、ダウンロード約3MB。GPU は不要です。


## Environment setup / 環境セットアップ

Colab's preinstalled scientific stack already provides everything needed (pandas, numpy, scipy, matplotlib). We only verify versions, so the run is reproducible and any missing dependency fails loudly instead of silently.
使用するライブラリはすべて Colab に標準搭載されています。バージョンを確認するだけです。

In [ ]:
import sys, math, time
import numpy as np
import pandas as pd
import scipy
import matplotlib
import matplotlib.pyplot as plt

print("python:", sys.version.split()[0])
for mod in (np, pd, scipy, matplotlib):
    print(f"{mod.__name__}: {mod.__version__}")

## Acquire the authors' results table / 著者公開の結果表の取得

`TableS1.csv` is the authors' published analysis output ("results from the analysis", repo README) — 5724 rows, one per side-view fruit image, with per-image linear measurements (cm), PCA scores, REML-adjusted trait predictions, harvest weight and release year. It is identical in role to the bioRxiv supporting Table S1 (media-3.csv).

We pin the exact commit and **verify integrity with the git blob SHA-1** (computed as `sha1("blob <size>\0" + bytes)`, the same hash GitHub reports for the file), so a silent change upstream cannot enter the analysis. Fallback: the same table from bioRxiv DC3 (media-3.csv).
`TableS1.csv` を pinned コミットから取得し、git blob SHA-1 で完全性を検証します（フォールバック: bioRxiv DC3 の同一表）。

In [ ]:
import hashlib
import urllib.request

REPO = "kylidevi/apple_shape"
COMMIT = "5bf0df08c993d1c59cdcdd52435b5185ecada11a"  # pinned revision
TABLES1_SIZE = 2_689_744
TABLES1_BLOB_SHA1 = "95802f08bf4e0a339aed239ba3b8eb9b12068ccd"
UA = {"User-Agent": "colab-phenotype-reproduction"}

def fetch(url, timeout=180):
    req = urllib.request.Request(url, headers=UA)
    return urllib.request.urlopen(req, timeout=timeout).read()

raw = None
source_used = None
try:
    raw = fetch(f"https://raw.githubusercontent.com/{REPO}/{COMMIT}/TableS1.csv")
    source_used = f"github.com/{REPO} @ {COMMIT} (raw)"
except Exception as e:
    print("GitHub raw failed:", type(e).__name__, e)

if raw is None:
    raw = fetch("https://www.biorxiv.org/content/biorxiv/early/2025/01/02/2025.01.01.631017/DC3/embed/media-3.csv?download=true")
    source_used = "biorxiv.org DC3 media-3.csv (Table S1)"
    assert len(raw) > 1_000_000 and not raw.lstrip().startswith(b"<"), "bioRxiv fallback returned unexpected bytes"
else:
    assert len(raw) == TABLES1_SIZE, f"size {len(raw)} != {TABLES1_SIZE}"
    blob_sha1 = hashlib.sha1(b"blob %d\x00" % len(raw) + raw).hexdigest()
    print("git blob sha1:", blob_sha1)
    assert blob_sha1 == TABLES1_BLOB_SHA1, "blob SHA-1 mismatch - refusing to use"

with open("/content/TableS1.csv", "wb") as f:
    f.write(raw)
print("source:", source_used)
print("bytes:", len(raw))

## Load and verify the table / 表の読み込みと検証

Expected structure (verified against the pinned commit and the bioRxiv Table S1 caption): 5724 images (rows) × 35 columns; each row is one side-view image with its `nursery_id` (tree), `apple_id` (accession/cultivar), manually placed `top_*`/`bottom_*` landmark pixels, dimensions in cm, raw and REML-adjusted PCA scores, and cultivar metadata. The 10 traits analysed in the paper (45 pairwise correlations) map to these columns:
Width→`width_reml_lsmeans_predict`, Length→`length_reml_lsmeans_predict`, Area→`area_reml_lsmeans_predict`, Solidity→`solidity_reml_lsmeans_predict`, Aspect Ratio→`aspect_ratio`, Harvest Weight→`weight_avg_16_harv`, PC1/PC2/PC3→`PC{1,2,3}_reml_lsmeans_predict`, Release Year→`release_year`. (The repo's later 11-trait variant adds `circularity_reml_lsmeans_predict`; the bioRxiv v1 supplement we validate against has 10 traits.)
表を読み込み、行数・列数・アクセッション数を検証します。

In [ ]:
table = pd.read_csv("/content/TableS1.csv")
print("shape:", table.shape)
assert table.shape == (5724, 35), "unexpected table shape"
n_accessions = table["apple_id"].nunique()
n_trees = table["nursery_id"].nunique()
print("unique apple_id (accessions):", n_accessions)
print("unique nursery_id (trees):", n_trees)
assert n_accessions == 534, "expected 534 accessions"
assert table["species"].eq("domestica").all(), "non-domestica rows present"
print("px.cm (px per cm scale):", table["px.cm"].unique())

## Input preview / 入力データのプレビュー

A quick look at the actual data the reproduction consumes: the per-image trait distributions. Aspect ratio ≈ width/length (oblate fruit <1, oblong fruit >1); harvest weight is the 2016 average per accession (g). This preview uses the author-published columns directly — no images are needed for this reproduction, and none are shown.
再現で使うデータ（5724画像分の形質分布）を確認します。この再現では画像入力は使用しません。

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
axes[0].hist(table["aspect_ratio"], bins=60, color="tab:blue")
axes[0].set_xlabel("aspect ratio (width / length)")
axes[0].set_ylabel("images")
axes[0].set_title("Per-image aspect ratio (5724 images)")
axes[1].hist(table["weight_avg_16_harv"], bins=60, color="tab:green")
axes[1].set_xlabel("harvest weight 2016 (g)")
axes[1].set_title("Harvest weight")
axes[2].scatter(table["width"], table["length"], s=1, alpha=0.15, color="tab:orange")
axes[2].set_xlabel("width (cm)")
axes[2].set_ylabel("length (cm)")
axes[2].set_title("Per-image width vs length")
plt.tight_layout()
plt.show()
display(table[["file", "nursery_id", "apple_id", "width", "length", "aspect_ratio",
               "PC1_reml_lsmeans_predict", "release_year"]].head(5))

## Method: from the authors' R script to this notebook / メソッド対応表

The authors' `TableS2.R` (commit `5bf0df0`) computes the correlation table from `analysis_apple_metadata.csv` (= the published `TableS1.csv` content) exactly as follows; this is the specification this notebook implements. The published bioRxiv v1 Table S2 has 10 traits (45 pairs); the repo script's 11-trait variant (55 pairs, adds circularity) belongs to the revised version and is noted but not validated here.

| `TableS2.R` (R 4.4) | This notebook (Python) | Note |
| --- | --- | --- |
| `read.csv(...) %>% distinct(apple_id, .keep_all = TRUE)` | `table.drop_duplicates(subset="apple_id", keep="first")` | one row per accession (534) |
| `dplyr::select(18:27, 35)` | explicit 10-trait column mapping (see above, minus `circularity_reml`) | cols 18:27+35 in the R file |
| `cor.test(x, y, method="spearman")$estimate` | `scipy.stats.spearmanr(x, y).statistic` on pairwise-complete rows | identical rank statistic |
| `cor.test(...) $p.value` | `spearmanr(x, y).pvalue` | R and SciPy use different asymptotic approximations for large n — p-values may differ slightly at the 3rd–4th decimal; the rank statistic rho must match tightly |
| `p.adjust(..., method="bonferroni")` over the 45 comparisons | `min(p × 45, 1)` | identical correction |

**Missing values:** 7 of the 534 accessions have no harvest weight and 326 have no release year (the value is absent from *all* of that accession's rows, so it is genuinely unrecorded, not a row-selection artifact). Each correlation is therefore computed on **pairwise-complete observations** (n = 534, 527 or 208 depending on the pair). This matches the authors' published rho values exactly (verified to ~1e-15 in the comparison below), and the pair sample size is reported alongside each result.
**欠測値:** 収穫重量7、 release year 326 アクセッションは値が全行で欠落しているため、ペアごとに欠測を除外して計算します（公開値と一致）。

## Build the accession-level trait matrix / アクセッション単位の形質行列

Following `TableS2.R`, keep the **first row per `apple_id`** (`distinct(..., .keep_all=TRUE)`) → 534 accessions, and assemble the 10 published traits. Values are the authors' REML-adjusted least-squares means (orchard position corrected by the authors with `lme4`; we use their published numbers rather than refitting). Missing values are expected and reported, not an error: some accessions have no recorded harvest weight or release year.
`TableS2.R` に従い、`apple_id` ごとの最初の行（534アクセッション）で10形質の行列を作ります。欠測はそのまま報告します。

In [ ]:
TRAIT_COLS = {
    "Width": "width_reml_lsmeans_predict",
    "Length": "length_reml_lsmeans_predict",
    "Area": "area_reml_lsmeans_predict",
    "Solidity": "solidity_reml_lsmeans_predict",
    "Aspect Ratio": "aspect_ratio",
    "Harvest Weight": "weight_avg_16_harv",
    "PC1": "PC1_reml_lsmeans_predict",
    "PC2": "PC2_reml_lsmeans_predict",
    "PC3": "PC3_reml_lsmeans_predict",
    "Release Year": "release_year",
}

acc = table.drop_duplicates(subset="apple_id", keep="first")
print("accessions:", len(acc))
traits = acc[list(TRAIT_COLS.values())].copy()
traits.columns = list(TRAIT_COLS.keys())
n_missing = traits.isna().sum()
print("accessions with missing values per trait:")
print(n_missing[n_missing > 0])
assert n_missing.sum() > 0 and set(n_missing.index[n_missing > 0]) <= {"Harvest Weight", "Release Year"}, \
    "unexpected missing-value pattern"
display(traits.describe().T[["count", "mean", "std", "min", "max"]])

## Recompute the 45 Spearman correlations + Bonferroni correction / Spearman相関とBonferroni補正の再計算

All 45 pairwise Spearman rank correlations among the 10 traits, computed on **pairwise-complete rows** for each pair (some accessions lack harvest weight or release year), then a Bonferroni correction across exactly 45 comparisons — the paper v1 design (the published v1 supplement has 45 pairs). The per-pair sample size is kept so the correction and comparison are transparent.
10形質間の全45ペアの Spearman 順相関をペアごとに欠測を除外して計算し、45比較に対する Bonferroni 補正を適用します。

In [ ]:
from scipy.stats import spearmanr

names = list(TRAIT_COLS.keys())
rows = []
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        t1, t2 = names[i], names[j]
        pair = traits[[t1, t2]].dropna()
        rho, p = spearmanr(pair[t1], pair[t2])
        rows.append({"trait 1": t1, "trait 2": t2,
                     "rho": float(rho), "p_raw": float(p), "n": len(pair)})
corr = pd.DataFrame(rows)
corr["p_bonferroni"] = np.minimum(corr["p_raw"] * len(corr), 1.0)
print("comparisons:", len(corr))
n_sig = int((corr["p_bonferroni"] < 0.05).sum())
print("significant after Bonferroni (p < 0.05):", n_sig, "of", len(corr))
print("pair sample sizes:", sorted(corr["n"].unique()))
display(corr.head(10))

## Fetch the authors' published Table S2 / 公開済み Table S2 の取得

The comparison target: bioRxiv Online Supporting Information **Table S2** (`media-4.csv`, 45 rows: `trait 1`, `trait 2`, `correlation`, `pval` — Bonferroni-corrected p-values). bioRxiv rate-limits automated clients (HTTP 429 with `Retry-After` was observed during development), so this cell retries a bounded number of times, honouring `Retry-After`.
比較対象として、bioRxiv 補足 Table S2（media-4.csv、45行）を取得します。429 の場合は Retry-After を守って有限回リトライします。

In [ ]:
import io
import requests

S2_URL = ("https://www.biorxiv.org/content/biorxiv/early/2025/01/02/"
          "2025.01.01.631017/DC4/embed/media-4.csv?download=true")
content = None
for attempt in range(5):
    r = requests.get(S2_URL, timeout=120, headers=UA)
    ct = r.headers.get("Content-Type", "")
    print("attempt", attempt, "->", r.status_code, ct, len(r.content), "bytes")
    if r.status_code == 200 and not r.content.lstrip().startswith(b"<"):
        content = r.content
        break
    time.sleep(min(int(r.headers.get("Retry-After", 30)), 60))
assert content is not None, "bioRxiv rate limit persisted after bounded retries"

with open("/content/TableS2_published.csv", "wb") as f:
    f.write(content)
s2_pub = pd.read_csv(io.BytesIO(content))
print("published Table S2 shape:", s2_pub.shape)
assert list(s2_pub.columns) == ["trait 1", "trait 2", "correlation", "pval"]
display(s2_pub.head(5))

## Validate: recomputed vs published / 再計算値と公開値の照合

Each published row is matched to our recomputation by its (unordered) trait pair. The **rank statistic rho must agree tightly** (published values are rounded to 6 decimals; the recomputation reproduces them essentially exactly). The **p-values are compared with a looser tolerance** because R's `cor.test` and SciPy use different asymptotic approximations — a few near-boundary p-values differ by ~1e-3. We report the worst-case deviations and fail loudly if the match is not close.
公開 Table S2 と再計算結果をペアごとに照合します（rho は厳密一致、p 値は R/SciPy の近似差を考慮した許容誤差）。

In [ ]:
def pair_key(a, b):
    return tuple(sorted([str(a).strip(), str(b).strip()]))

pub = s2_pub.copy()
pub["key"] = [pair_key(a, b) for a, b in zip(pub["trait 1"], pub["trait 2"])]
ours = corr.copy()
ours["key"] = [pair_key(a, b) for a, b in zip(ours["trait 1"], ours["trait 2"])]
merged = pub.merge(ours, on="key", suffixes=("_pub", "_ours"), how="outer", indicator=True)
assert merged["_merge"].eq("both").all(), "trait-pair mismatch against published table"
assert len(merged) == 45

merged["drho"] = (merged["correlation"] - merged["rho"]).abs()
merged["dp"] = (merged["pval"] - merged["p_bonferroni"]).abs()
print(f"max |Δ rho|        = {merged['drho'].max():.2e}")
print(f"max |Δ p (bonf.)  | = {merged['dp'].max():.2e}")
rho_ok = merged["drho"].max() < 5e-4
p_ok = merged["dp"].max() < 5e-3
print("rho agreement (tol 5e-4):", rho_ok)
print("p agreement  (tol 5e-3):", p_ok)
assert rho_ok and p_ok, "recomputation does not match the published Table S2"
n_p_diff = int((merged["dp"] > 1e-3).sum())
print(f"pairs with |Δp| > 1e-3 (R vs SciPy approximation): {n_p_diff}")
display(merged.loc[merged["dp"] > 1e-3, ["trait 1_pub", "trait 2_pub", "pval", "p_bonferroni"]])

## Headline claims of the paper / 論文の主要結果の確認

The paper's key statements (RESULTS): PC1–aspect ratio is the strongest of all 45 correlations (rho = −0.964, p < 1×10⁻¹⁵); PC1 is **not** significantly correlated with fruit size (area: rho = 0.126, p = 0.162; harvest weight: rho = 0.066, p = 1.000 after correction). We check our recomputation against these exact numbers and against the published table rows.
論文の主要な数値（PC1×アスペクト比 rho=−0.964、PC1×面積・重量は有意なし）を再計算値と突き合わせます。

In [ ]:
def ours_row(t1, t2):
    k = pair_key(t1, t2)
    row = corr[[pair_key(a, b) == k for a, b in zip(corr["trait 1"], corr["trait 2"])]]
    return row.iloc[0]

checks = [
    ("PC1", "Aspect Ratio", -0.964333, "shape tracks aspect ratio (strongest of 45)"),
    ("PC1", "Area", 0.125884, "PC1 vs size: NOT significant (paper: p=0.162)"),
    ("PC1", "Harvest Weight", 0.065890, "PC1 vs size: NOT significant (paper: p=1.000)"),
]
for t1, t2, pub_rho, label in checks:
    r = ours_row(t1, t2)
    print(f"{t1} x {t2:15s}: ours rho={r['rho']:+.6f} p_bonf={r['p_bonferroni']:.6f} | "
          f"published rho={pub_rho:+.6f} | {label}")
r_ar = ours_row("PC1", "Aspect Ratio")
assert r_ar["rho"] < -0.96 and r_ar["p_bonferroni"] < 1e-15
r_area = ours_row("PC1", "Area")
assert abs(r_area["rho"] - 0.126) < 5e-4 and r_area["p_bonferroni"] > 0.05
r_wt = ours_row("PC1", "Harvest Weight")
assert abs(r_wt["rho"] - 0.066) < 5e-4 and r_wt["p_bonferroni"] > 0.99
print("headline claims reproduced: shape independent of size")

## Figure-3-style morphospace / 形態空間（Figure 3 スタイル）

The paper visualises the shape space as PC1 vs PC3, one point per accession, colored by aspect ratio. We render the same view from the authors' published REML-adjusted PC scores (`PC{1,3}_reml_lsmeans_predict`), reusing the authors' colormap and normalization range (viridis, 0.86–1.28, from `apple_morphometric.qmd`). **Difference from the paper's Figure 3:** the grey eigen-apples (inverse PCA reconstructions) are omitted — they require the image-based PCA model, which is out of scope here. This is our rendering of author-published data, not the author's figure.
論文 Figure 3 と同じ PC1×PC3 空間を、著者公開の REML 調整済み PC スコア・同じカラーマップ/正規化範囲で描画します（eigen-apple は範囲外のため省略）。

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
sc = ax.scatter(acc["PC1_reml_lsmeans_predict"], acc["PC3_reml_lsmeans_predict"],
                c=acc["aspect_ratio"], cmap="viridis", vmin=0.86, vmax=1.28,
                s=20, linewidth=0)
cbar = plt.colorbar(sc, ax=ax)
cbar.set_ticks(np.linspace(0.86, 1.28, 7))
cbar.set_label("Aspect Ratio")
ax.set_xlabel("PC1 (shape: aspect ratio axis), REML-adjusted")
ax.set_ylabel("PC3 (shape: asymmetry axis), REML-adjusted")
ax.set_title("PC1-PC3 morphospace, 534 accessions\n(our rendering of author-published scores; eigen-apples omitted)")
ax.grid(alpha=0.4)
plt.tight_layout()
plt.show()

## Export the recomputed correlation table / 再計算した相関表のエクスポート

Save our recomputed 45-row table as CSV in the Colab workspace (`/content/`), alongside the published one for side-by-side inspection.
再計算した45行の相関表を CSV として保存し、公開表と並べて表示します。

In [ ]:
export_cols = ["trait 1", "trait 2", "rho", "p_raw", "p_bonferroni"]
corr_export = corr[export_cols].rename(columns={
    "rho": "correlation", "p_raw": "pval_raw", "p_bonferroni": "pval_bonferroni"})
corr_export.to_csv("/content/recomputed_TableS2.csv", index=False)
print("saved /content/recomputed_TableS2.csv")

side_by_side = merged[["trait 1_pub", "trait 2_pub", "correlation", "rho",
                       "pval", "p_bonferroni"]].sort_values(["trait 1_pub", "trait 2_pub"])
display(side_by_side.reset_index(drop=True))

## Interpretation and validation record / 解釈と検証記録

**What was verified in this run:**
- The 45-pair Bonferroni-corrected Spearman matrix recomputed from the authors' published per-image table matches the authors' published Table S2 (bioRxiv media-4.csv): all 45 rank correlations agree to the published 6-decimal rounding (max |Δrho| ≈ 1e-15), and Bonferroni-corrected p-values agree within ~1e-3 (largest deviations on near-boundary pairs, from the documented R/SciPy asymptotic difference). 20 of 45 comparisons are significant in both our recomputation and the published table.
- Headline result: shape PC1 ↔ aspect ratio rho ≈ −0.964 (p ≪ 0.001, strongest of 45); PC1 ↔ area and PC1 ↔ harvest weight are **not significant** — fruit shape varies independently of fruit size, the paper's central claim.
- Missing-value structure: 7 accessions lack harvest weight and 326 lack release year; pairwise-complete correlation sizes (534/527/208) reproduce the published statistics exactly.

**Known differences / 既知の差分:**
- p-values come from SciPy's asymptotic approximation rather than R's `cor.test` — differences of up to ~1.3e-3 on three near-boundary pairs (shown in the comparison cell output).
- The repo's `TableS2.R` at the pinned commit implements an 11-trait (55-comparison) variant adding circularity — that belongs to the revised article; the bioRxiv v1 supplement validated here has 10 traits / 45 comparisons.
- The image pipeline (99 pseudo-landmarks, GPA, PCA, eigen-apples) and the `lme4` REML adjustment are **not** re-executed; their published outputs are used as inputs. The Dryad binary-image archive could not be retrieved through authorized anonymous routes (API download requires a Dryad bearer token; the UI route serves an anti-bot challenge) — recorded as an access limitation, not missing material.
- A single-table statistical reproduction does not re-verify the full paper; it verifies the correlation analysis supporting its headline claim.

**日本語:** 公開 Table S2 との照合により、45ペアすべての順位相関が公開値と一致し（最大差 ≈1e-15）、主要結果（PC1↔アスペクト比 rho≈−0.964、PC1↔サイズは無相関、45ペア中20ペアが有意）が再現されました。p 値は R/SciPy の近似差により一部 ~1e-3 程度の差があります。画像処理・GPA/PCA・REML 再推定は範囲外（Dryad 画像は自動取得不可）。

## References / 参考資料

- DiVilly, K. et al. *Differences in apple fruit shape are independent of fruit size.* bioRxiv 2025.01.01.631017 (v1, 2025-01-02). https://doi.org/10.1101/2025.01.01.631017 — journal version: https://doi.org/10.1002/ppj2.70032
- Author code: https://github.com/kylidevi/apple_shape @ `5bf0df08c993d1c59cdcdd52435b5185ecada11a` — `TableS2.R` (correlation analysis), `apple_morphometric.qmd` (image pipeline, morphospace parameters), `Figure2&5.R` (figure specifications).
- Author data: Dryad https://doi.org/10.5061/dryad.xgxd254tg (v3, CC0-1.0; `apple_metadata.csv`, `all_binaries.zip`); bioRxiv supporting tables: Table S1 `DC3/embed/media-3.csv`, Table S2 `DC4/embed/media-4.csv`.
- Related: Watts et al. (2021) — Apple Biodiversity Collection design and REML adjustment method referenced by the paper.
- This notebook was AI-authored for the PhenoPaper reproduction pipeline; correlation code is an AI-assisted R→Python port of `TableS2.R` as disclosed above.